In [1]:
import json 
import requests

from dotenv import load_dotenv
import os

In [2]:
def load_api_key():
    """
    Load Steam API keys from the .env file located in the .venv folder.
    
    Returns:
        tuple: A tuple containing (access_token, web_api_key).
    """
    # Notebook is in the 'src/' folder, so go up one level to reach '.venv/.env'
    env_path = "../.venv/.env" 
    
    load_dotenv(dotenv_path=env_path)
    access_token = os.getenv('access_token')
    web_api_key = os.getenv('web_api_key')
    return access_token, web_api_key

access_token, web_api_key = load_api_key()

In [ ]:
import time

def get_apps_id(access_token):
    """
    Fetch the list of all Steam games and their IDs using pagination.
    
    Returns:
        dict: A combined JSON response containing the list of all apps if successful.
    """
    all_apps = []
    last_appid = 0
    have_more_results = True
    
    while have_more_results:
        # Steam API uses 'key=' instead of 'access_token='
        url = f"https://api.steampowered.com/IStoreService/GetAppList/v1/?key={access_token}&last_appid={last_appid}&max_results=10000"
        
        response = requests.get(url)
        if response.status_code == 200:
            games_data = response.json()
            response_data = games_data.get("response", {})
            
            apps_batch = response_data.get("apps", [])
            all_apps.extend(apps_batch)
            
            have_more_results = response_data.get("have_more_results", False)
            last_appid = response_data.get("last_appid", last_appid)
            
            # Optional: Sleep briefly to avoid hitting rate limits too quickly
            time.sleep(0.5)
        else: 
            print(f"error: {response.status_code}")
            break
            
    # Structure the final combined output similar to the original single response
    final_output = {
        "response": {
            "apps": all_apps,
            "have_more_results": False,
            "last_appid": last_appid
        }
    }
    
    output_path = "../data/games_id_all.json"
    with open(output_path, "w") as f:
        json.dump(final_output, f)
    
    return final_output

# Use web_api_key instead of access_token
games_id = get_apps_id(web_api_key)
print(f"Total games fetched: {len(games_id['response']['apps'])}")

Fetching games starting after App ID: 0...
Fetching games starting after App ID: 507380...
Fetching games starting after App ID: 779590...
Fetching games starting after App ID: 1046630...
Fetching games starting after App ID: 1289900...
Fetching games starting after App ID: 1537980...
Fetching games starting after App ID: 1783490...
Fetching games starting after App ID: 2051500...
Fetching games starting after App ID: 2296580...
Fetching games starting after App ID: 2542140...
Fetching games starting after App ID: 2806830...
Fetching games starting after App ID: 3064020...
Fetching games starting after App ID: 3326050...
Fetching games starting after App ID: 3593870...
Fetching games starting after App ID: 3869090...
Fetching games starting after App ID: 4154970...
Fetching games starting after App ID: 4482970...
Total games fetched: 162941


### Using game id, we fetch informations about it from steam server

In [9]:
def get_games_raw_data(app_id):
    """
    Fetch detailed store information for a specific game by its App ID.
    Saves the raw JSON response to a local file and returns the parsed game data.
    
    Args:
        app_id (int or str): The Steam Application ID to fetch data for.
        
    Returns:
        dict: The game's store data details.
    """
    url = f"https://store.steampowered.com/api/appdetails?appids={app_id}"
    response = requests.get(url)
    
    data = None
    game_name = None
    
    if response.status_code == 200:
        try:
            data = response.json()
            str_app_id = str(app_id)
            game_name = data[str_app_id]["data"]["name"]
            output_path = f"../data/games_informations/{game_name}_{app_id}.json"
            # with open(output_path, "w") as f:
            #     json.dump(data, f)
        except KeyError:
            print(f"Skipping {app_id}: No valid store data or name found.")

    else:
        print(f"Failed to fetch {app_id}. Status code: {response.status_code}")
        
    return data, game_name

In [10]:
def structurize_game_output(raw: dict, gamename):
    import re
    first_key = list(raw.keys())[0]
    if isinstance(raw.get(first_key), dict) and "data" in raw[first_key]:
        raw_data = raw[first_key]["data"]
    else:
        raw_data = raw
        
    to_pop = ["type", "steam_appid", "required_age", "detailed_description", "short_description",
              "capsule_image", "capsule_imagev5", "website", "pc_requirements","mac_requirements",
              "packages", "package_groups", "linux_requirements", "platforms", "metacritic", 
              "screenshots", "background", "background_raw", "content_descriptors", "ratings",
              "movies", "legal_notice"]        
    for att in to_pop:
        raw_data.pop(att, None)

    to_pop_price_overview = ["initial_formatted", "final_formatted"]
    if "price_overview" in raw_data:
        for att in to_pop_price_overview:
            if att in raw_data["price_overview"]:
                raw_data["price_overview"].pop(att)
    
    if not raw_data.get("achievements"):
        raw_data["achievements"] = None
    
    raw_data["game_link"] = f"https://store.steampowered.com/app/{first_key}/{gamename}/"

    structurized = {}
    structurized[first_key] = raw_data
        
    safe_gamename = re.sub(r'[\\/*?:"<>|]', "", gamename)
    output_path = f"../data/games_informations/{first_key}_{safe_gamename}.json"
    
    with open(output_path, "w") as f:
        json.dump(structurized, f)

In [ ]:
all_ids = []
for app in games_id["response"]["apps"]:
    all_ids.append(app["appid"])

# 1. Read existing files to build a set of already downloaded App IDs
output_dir = "../data/games_informations/"
os.makedirs(output_dir, exist_ok=True) # Ensure folder exists
existing_files = os.listdir(output_dir)

existing_app_ids = set()
for fname in existing_files:
    if fname.endswith(".json"):
        # Files are named like {appid}_{safegamename}.json
        parts = fname.split("_", 1)
        if parts[0].isdigit():
            existing_app_ids.add(int(parts[0]))

print(f"Found {len(existing_app_ids)} already fetched games in directory.")

# 2. Fetch missing ones
for n in all_ids:
    if n in existing_app_ids:
        # Already downloaded, skip to next without hitting the API
        continue
        
    rawww, gamename = get_games_raw_data(n)
    
    # Critical: Sleep to respect Steam's ~200 requests / 5 min limit (approx 1.5 sec per request)
    time.sleep(1.5) 
    
    if gamename is not None:
        structurize_game_output(rawww, gamename)
        print(f"Saved json for {gamename} (AppID: {n})")
        
        # Add to set so we don't fetch it again
        existing_app_ids.add(n)
    else:
        # It's up to you if you want to store "failed/dead" app IDs in another 
        # text file to skip them completely on future runs.
        pass

saved json for Counter-Strike
saved json for Team Fortress Classic
saved json for Day of Defeat


KeyboardInterrupt: 